# TN3K Thyroid Nodule — Boundary Loss Ablation

비교: `ce_dice` / `plwce_dice` (baseline) vs Boundary Loss 변형 8종

In [ ]:
# === Cell 0: 환경 설정 ===
import subprocess,sys
for pkg in ['segmentation-models-pytorch','openpyxl','albumentations','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
import os,warnings,json,random,glob,shutil
warnings.filterwarnings('ignore')
os.environ['TQDM_DISABLE']='1'
import numpy as np,cv2
from tqdm import tqdm
import torch,torch.nn as nn,torch.optim as optim
from torch.utils.data import Dataset,DataLoader
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score
import segmentation_models_pytorch as smp
_CL_LOCAL='/root/imbalanced-data-LWCE/medical_data'
_CL_COLAB='/tmp/custom_losses'
sys.path.insert(0,_CL_LOCAL if os.path.exists(_CL_LOCAL) else _CL_COLAB)
from custom_losses import get_loss_function
GDRIVE_DATA_PATH='/content/drive/MyDrive/imbalanced-data-LWCE/tn3k'
IS_COLAB=False
try:
    from google.colab import drive; drive.mount('/content/drive'); IS_COLAB=True
    _src='/content/drive/MyDrive/imbalanced-data-LWCE/medical_data/custom_losses.py'
    if not os.path.exists(os.path.join(_CL_LOCAL,'custom_losses.py')) and os.path.exists(_src):
        os.makedirs(_CL_COLAB,exist_ok=True); shutil.copy(_src,_CL_COLAB)
    print('Google Drive 마운트 완료')
except Exception: print('Colab 환경 아님')
DOMAIN='tn3k'; NUM_CLASSES=2; CLASS_NAMES=['Background','Nodule']
IMG_SIZE=256; BATCH_SIZE=16; NUM_WORKERS=0; SEED=42
MEAN=np.array([0.485,0.456,0.406],dtype=np.float32)
STD=np.array([0.229,0.224,0.225],dtype=np.float32)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
RESULTS_DIR='/root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/tn3k'
os.makedirs(RESULTS_DIR,exist_ok=True)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}'); print('환경 설정 완료')

In [ ]:
# === Cell 1: 데이터 로드 ===
TMP_BASE='/tmp/tn3k_data'
if IS_COLAB and os.path.exists(GDRIVE_DATA_PATH):
    img_check=glob.glob(os.path.join(TMP_BASE,'tg3k','thyroid-image','*.jpg'))
    if len(img_check)<100:
        print('Google Drive에서 복사 중...')
        os.makedirs(TMP_BASE,exist_ok=True)
        shutil.copytree(GDRIVE_DATA_PATH,TMP_BASE,dirs_exist_ok=True)
        print('복사 완료')
    else: print(f'캐시 사용: {len(img_check)}개')
    BASE_DIR=TMP_BASE
else:
    raise RuntimeError('Google Drive 마운트 필요. Cell 0 재실행.
경로: '+GDRIVE_DATA_PATH)
TN3K_TRAIN_IMG=os.path.join(BASE_DIR,'tn3k','trainval-image')
TN3K_TRAIN_MASK=os.path.join(BASE_DIR,'tn3k','trainval-mask')
TEST_IMG=os.path.join(BASE_DIR,'tn3k','test-image')
TEST_MASK=os.path.join(BASE_DIR,'tn3k','test-mask')
FOLD_JSON=os.path.join(BASE_DIR,'tn3k','tn3k-trainval-fold0.json')
with open(FOLD_JSON) as f: fold=json.load(f)
def idx_to_path(idx,id,md):
    fn=f'{idx:04d}.jpg'; return os.path.join(id,fn),os.path.join(md,fn)
tr_imgs,tr_masks,val_imgs,val_masks=[],[],[],[]
for idx in fold['train']:
    ip,mp=idx_to_path(idx,TN3K_TRAIN_IMG,TN3K_TRAIN_MASK)
    if os.path.exists(ip) and os.path.exists(mp): tr_imgs.append(ip);tr_masks.append(mp)
for idx in fold['val']:
    ip,mp=idx_to_path(idx,TN3K_TRAIN_IMG,TN3K_TRAIN_MASK)
    if os.path.exists(ip) and os.path.exists(mp): val_imgs.append(ip);val_masks.append(mp)
test_imgs=sorted(glob.glob(os.path.join(TEST_IMG,'*.jpg')))
test_masks=sorted(glob.glob(os.path.join(TEST_MASK,'*.jpg')))
print(f'Train:{len(tr_imgs)} Val:{len(val_imgs)} Test:{len(test_imgs)}')
_M=MEAN.tolist();_S=STD.tolist()
train_tf=A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE*3),
    A.PadIfNeeded(min_height=IMG_SIZE*3,min_width=IMG_SIZE*3,border_mode=0,value=0,mask_value=0),
    A.RandomResizedCrop(size=(IMG_SIZE,IMG_SIZE),scale=(0.05,1.0),ratio=(0.5,2.0)),
    A.HorizontalFlip(p=0.5),A.VerticalFlip(p=0.5),A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(0.05,0.15,15,border_mode=0,p=0.5),
    A.ElasticTransform(alpha=60,sigma=6,p=0.3),
    A.RandomBrightnessContrast(0.3,0.3,p=0.7),A.GaussNoise((10.0,50.0),p=0.5),
    A.Normalize(mean=_M,std=_S),ToTensorV2()])
val_tf=A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE,min_width=IMG_SIZE,border_mode=0,value=0,mask_value=0),
    A.Normalize(mean=_M,std=_S),ToTensorV2()])
class TN3KDataset(Dataset):
    def __init__(self,ip,mp,tf=None): self.ip=ip;self.mp=mp;self.tf=tf
    def __len__(self): return len(self.ip)
    def __getitem__(self,idx):
        img=cv2.imread(self.ip[idx]); mask=cv2.imread(self.mp[idx],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE,3),dtype=np.uint8)
        if mask is None: mask=np.zeros((IMG_SIZE,IMG_SIZE),dtype=np.uint8)
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB); mask=(mask>128).astype(np.uint8)
        if self.tf:
            aug=self.tf(image=img,mask=mask); img,mask=aug['image'],aug['mask'].long()
        else:
            img=cv2.resize(img,(IMG_SIZE,IMG_SIZE))
            img=(img.astype(np.float32)/255.0-MEAN)/STD
            img=torch.from_numpy(img.transpose(2,0,1).astype(np.float32))
            mask=torch.from_numpy(mask.astype(np.int64))
        return img,mask
train_loader=DataLoader(TN3KDataset(tr_imgs,tr_masks,train_tf),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
val_loader=DataLoader(TN3KDataset(val_imgs,val_masks,val_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
test_loader=DataLoader(TN3KDataset(test_imgs,test_masks,val_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
print('DataLoader 완료')

In [ ]:
# === Cell 2: 클래스 비율 계산 ===
print('클래스 비율 계산 중...')
class_counts=np.zeros(NUM_CLASSES,dtype=np.int64)
for mp in tqdm(tr_masks):
    mask=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
    class_counts[1]+=int((mask>128).sum()); class_counts[0]+=int((mask<=128).sum())
class_counts=class_counts.tolist()
print(f'BG:{class_counts[0]:,}  Nodule:{class_counts[1]:,}  비율:{class_counts[0]/class_counts[1]:.1f}:1')
print(f'class_counts={class_counts}')

In [ ]:
# === Cell 3: 모델 정의 ===

def to_2ch_logits(p):
    return torch.cat([-p, p], dim=1)

def build_model():
    return smp.Unet(
        encoder_name='resnet34', encoder_weights='imagenet',
        in_channels=3, classes=1, activation=None,
    ).to(device)

def compute_val_dice(model, loader):
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0])
            pred = (prob > 0.5).long()
            tp += ((pred==1)&(masks==1)).sum().item()
            fp += ((pred==1)&(masks==0)).sum().item()
            fn += ((pred==0)&(masks==1)).sum().item()
    return float(2*tp/(2*tp+fp+fn+1e-8))

def compute_val_metrics(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0]).cpu().numpy()
            pred = (prob > 0.5).astype(np.int64)
            all_probs.append(prob.flatten())
            all_preds.append(pred.flatten())
            all_labels.append(masks.numpy().flatten())
    all_probs  = np.concatenate(all_probs)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    TP = ((all_preds==1)&(all_labels==1)).sum()
    FP = ((all_preds==1)&(all_labels==0)).sum()
    TN = ((all_preds==0)&(all_labels==0)).sum()
    FN = ((all_preds==0)&(all_labels==1)).sum()
    dice = 2*TP/(2*TP+FP+FN+1e-8)
    sens = TP/(TP+FN+1e-8)
    spec = TN/(TN+FP+1e-8)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0
    return {'Dice':float(dice),'Sensitivity':float(sens),'Specificity':float(spec),'AUC':float(auc)}

print('모델 + 유틸리티 함수 준비 완료')

In [ ]:
# === Cell 4: 학습 함수 ===

def train_model(loss_name, alpha=1.0, gamma=2.0, epochs=30, lr=1e-4,
                subset_ratio=1.0, tag=''):
    model     = build_model()
    optimizer = optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)
    name = f'{loss_name}_a{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag: name = f'{tag}_{name}'
    print(f"\n{'='*60}\n{name}  (epochs={epochs})\n{'='*60}")
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset, random.sample(range(len(train_loader.dataset)), n))
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader
    history = {'loss':[], 'val_dice':[]}
    best_dice = 0.0
    save_path = f'/tmp/ba_tn3k_{name}.pth'
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(to_2ch_logits(model(imgs)), masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_dice = compute_val_dice(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)
        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}', end='')
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
        print()
    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val Dice: {best_dice:.4f}')
    return model, history, best_dice

print('train_model() 준비 완료')

In [ ]:
# === Cell 5: Optuna — PLWCE alpha 탐색 (Boundary Loss 환경) ===
# Dice+BL 조합에서는 alpha 최적값이 기존 Dice 전용 실험과 다를 수 있음.
# 두 대표 손실 함수에 대해 별도 탐색:
#   plwce_dice_boundary  → alpha_with_dice    (plwce_dice_log_boundary에도 재사용)
#   plwce_boundary       → alpha_without_dice (plwce_log_boundary에도 재사용)
import optuna, traceback
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TQDM_DISABLE'] = '1'

ALPHA_LOW    = 1.0
ALPHA_HIGH   = 20.0
PROXY_EPOCHS = 5
PROXY_RATIO  = 0.15
N_TRIALS     = 20

def make_objective(loss_name):
    def objective(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, dice = train_model(loss_name, alpha=alpha,
                                     epochs=PROXY_EPOCHS, subset_ratio=PROXY_RATIO)
            return dice
        except Exception:
            traceback.print_exc()
            return None
    return objective

# --- plwce_dice_boundary ---
print('Optuna: plwce_dice_boundary ...')
study_pdb = optuna.create_study(direction='maximize')
study_pdb.optimize(make_objective('plwce_dice_boundary'), n_trials=N_TRIALS)
best_trials_pdb = [t for t in study_pdb.trials if t.value is not None]
best_alpha_with_dice = (
    best_trials_pdb[int(np.argmax([t.value for t in best_trials_pdb]))].params['alpha']
    if best_trials_pdb else 6.896
)
print(f'  best alpha (with Dice): {best_alpha_with_dice:.3f}')

# --- plwce_boundary (no Dice) ---
print('Optuna: plwce_boundary (no Dice) ...')
study_pb = optuna.create_study(direction='maximize')
study_pb.optimize(make_objective('plwce_boundary'), n_trials=N_TRIALS)
best_trials_pb = [t for t in study_pb.trials if t.value is not None]
best_alpha_without_dice = (
    best_trials_pb[int(np.argmax([t.value for t in best_trials_pb]))].params['alpha']
    if best_trials_pb else 6.896
)
print(f'  best alpha (no Dice):   {best_alpha_without_dice:.3f}')

# JSON 저장
optuna_data = {
    'plwce_dice_boundary': {'best_alpha': best_alpha_with_dice},
    'plwce_boundary':      {'best_alpha': best_alpha_without_dice},
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json'), 'w') as f:
    json.dump(optuna_data, f, indent=2)
print('Optuna 결과 저장 완료')

In [ ]:
# === Cell 6: Boundary Ablation 전체 학습 ===

FINAL_EPOCHS = 50
FINAL_LR     = 1e-4

# --- Optuna 결과 로드 (Cell 5 미실행 시 JSON fallback) ---
try:
    _ = best_alpha_with_dice
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json')) as f:
            d = json.load(f)
        best_alpha_with_dice    = d['plwce_dice_boundary']['best_alpha']
        best_alpha_without_dice = d['plwce_boundary']['best_alpha']
        print(f'Optuna 로드: with_dice={best_alpha_with_dice:.3f}, no_dice={best_alpha_without_dice:.3f}')
    except FileNotFoundError:
        best_alpha_with_dice    = 6.896
        best_alpha_without_dice = 6.896
        print(f'Optuna 미실행 → fallback alpha=6.896')

experiments = [
    # (loss_name,                  alpha,                   label)
    ('ce_dice',                    1.0,                     'CE+Dice                     [baseline]'),
    ('plwce_dice',                 best_alpha_with_dice,    f'PLWCE+Dice                  (α={best_alpha_with_dice:.3f}) [baseline]'),
    ('ce_dice_boundary',           1.0,                     'CE+Dice+BL                  [literature]'),
    ('plwce_dice_boundary',        best_alpha_with_dice,    f'PLWCE+Dice+BL               (α={best_alpha_with_dice:.3f})'),
    ('plwce_dice_log_boundary',    best_alpha_with_dice,    f'PLWCE+Dice+LBL              (α={best_alpha_with_dice:.3f})'),
    ('ce_dice_log_boundary',       1.0,                     'CE+Dice+LBL (=Dice+LBL)'),
    ('plwce_boundary',             best_alpha_without_dice, f'PLWCE+BL     (no Dice)       (α={best_alpha_without_dice:.3f})'),
    ('plwce_log_boundary',         best_alpha_without_dice, f'PLWCE+LBL    (no Dice)       (α={best_alpha_without_dice:.3f})'),
]

all_results = {}
for loss_name, alpha, label in experiments:
    model, history, best_dice = train_model(
        loss_name=loss_name, alpha=alpha,
        epochs=FINAL_EPOCHS, lr=FINAL_LR, tag='ba')
    all_results[label] = {
        'model':model, 'history':history, 'best_dice':best_dice,
        'loss_name':loss_name, 'alpha':alpha,
    }

print('\n' + '='*65)
print('[Boundary Ablation 요약 — Val Dice]')
print(f"{'Loss':<52} {'Val Dice':>9}")
print('-'*63)
for label, v in all_results.items():
    print(f"{label:<52} {v['best_dice']:>9.4f}")

In [ ]:
# === Cell 7: 평가 및 결과 저장 ===

COLORS = ['#4878D0','#EE854A','#6ACC65','#D65F5F','#B47CC7','#956CB4','#8C613C','#DC7EC0']

# --- 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for i, (label, v) in enumerate(all_results.items()):
    c = COLORS[i % len(COLORS)]
    ax1.plot(v['history']['loss'],     label=label[:35], color=c)
    ax2.plot(v['history']['val_dice'], label=label[:35], color=c)
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)
ax2.set_title('Val Dice');      ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_curves.png'), dpi=150)
plt.show()

# --- Test set 정량 평가 ---
print('\n[Test Set 정량 평가]')
print(f"{'Loss':<52} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 80)

final_results = {}
for label, v in all_results.items():
    m = compute_val_metrics(v['model'], test_loader)
    final_results[label] = m
    print(f"{label:<52} {m['Dice']:>7.4f} {m['Sensitivity']:>7.4f} {m['Specificity']:>7.4f} {m['AUC']:>7.4f}")

# --- 바 차트 ---
labels = list(final_results.keys())
dices  = [final_results[l]['Dice'] for l in labels]
idx    = sorted(range(len(dices)), key=lambda i: dices[i], reverse=True)
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(range(len(labels)), [dices[i] for i in idx],
              color=[COLORS[i%len(COLORS)] for i in range(len(labels))])
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([labels[i][:40] for i in idx], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Test Dice')
ax.set_title(f'Boundary Ablation — Test Dice ({DOMAIN})')
for bar, val in zip(bars, [dices[i] for i in idx]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_bar.png'), dpi=150)
plt.show()

# --- JSON 저장 ---
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.json'), 'w') as f:
    json.dump({l:{k:v for k,v in m.items()} for l,m in final_results.items()},
              f, indent=2, ensure_ascii=False)
print(f'결과 저장 완료: {RESULTS_DIR}')